# Abstract Results — Strategy Comparison

Run all cells top to bottom. The last cell prints your abstract numbers and a ready-to-use paragraph.

**Strategies compared:** Equal Weight · Naive Risk Parity · ERC Risk Parity · MPT Static · MPT Rolling

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.optimize as sco

warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries loaded OK")

In [ ]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
with open("../../markets.json") as f:
    config = json.load(f)

BASE = Path("../../data")
name = list(config["markets"].keys())[0]

rets = pd.read_csv(
    BASE / f"{name}_returns_monthly.csv",
    index_col=0, parse_dates=True
)

ASSETS = list(rets.columns)
N      = len(ASSETS)

RISK_FREE_ANNUAL  = 0.0351
RISK_FREE_MONTHLY = (1 + RISK_FREE_ANNUAL) ** (1/12) - 1

print(f"Market : {name}")
print(f"Period : {rets.index[0].date()} -> {rets.index[-1].date()}")
print(f"Assets : {ASSETS}")
print(f"Months : {len(rets)}")

In [ ]:
# ── Cell 3: Helper Functions ──────────────────────────────────────────────────

def performance_stats(ret_series, label):
    """Annualized return, vol, Sharpe (with Rf), max drawdown."""
    ann_return = ret_series.mean() * 12 * 100
    ann_vol    = ret_series.std()  * np.sqrt(12) * 100
    sharpe     = (ret_series.mean() - RISK_FREE_MONTHLY) * 12 / \
                 (ret_series.std() * np.sqrt(12))
    cum    = (1 + ret_series).cumprod()
    max_dd = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    return pd.Series({
        "Ann. Return %"  : round(ann_return, 2),
        "Ann. Vol %"     : round(ann_vol,    2),
        "Sharpe"         : round(sharpe,     3),
        "Max Drawdown %" : round(max_dd,     2),
    }, name=label)

def crisis_drawdown(ret_series, start, end):
    """Worst peak-to-trough drawdown within a date window."""
    window = ret_series.loc[start:end]
    if window.empty:
        return np.nan
    cum = (1 + window).cumprod()
    return round(((cum - cum.cummax()) / cum.cummax()).min() * 100, 2)

def avg_turnover(weights_df):
    """
    Average monthly turnover = mean of the sum of absolute weight changes.
    Static strategies (weights never change) correctly return 0%.
    Rolling MPT returns its actual monthly reshuffling level.
    """
    changes = weights_df.diff().abs().sum(axis=1).dropna()
    return round(changes.mean() * 100, 2)

print("Helper functions ready")

In [ ]:
# ── Cell 4: Equal Weight ──────────────────────────────────────────────────────
eq_w   = pd.Series(1 / N, index=ASSETS)
eq_ret = (rets * eq_w).sum(axis=1)
eq_w_df = pd.DataFrame(
    np.tile(eq_w.values, (len(rets), 1)),
    index=rets.index, columns=ASSETS
)
print("Equal Weight done")
print((eq_w * 100).round(1).to_string())

In [ ]:
# ── Cell 5: Naive Risk Parity ─────────────────────────────────────────────────
# Weight = inverse volatility. Low-vol assets get more weight.
# Static — weights set once from full history, never change.

vol      = rets.std()
naive_w  = (1 / vol) / (1 / vol).sum()
naive_ret = (rets * naive_w).sum(axis=1)
naive_w_df = pd.DataFrame(
    np.tile(naive_w.values, (len(rets), 1)),
    index=rets.index, columns=ASSETS
)
print("Naive Risk Parity done")
print((naive_w * 100).round(2).to_string())

In [ ]:
# ── Cell 6: ERC Risk Parity ───────────────────────────────────────────────────
# Each asset contributes equally to total portfolio risk.
# Accounts for correlations. Static — weights set once, never change.

def erc_weights(returns):
    cov = returns.cov().values
    n   = cov.shape[0]
    def objective(w):
        pv  = np.sqrt(w @ cov @ w)
        mrc = (cov @ w) / pv
        rc  = w * mrc
        return sum((rc[i] - rc[j])**2 for i in range(n) for j in range(n))
    res = sco.minimize(
        objective, np.ones(n) / n, method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1},
        options={"ftol": 1e-12, "maxiter": 1000}
    )
    return pd.Series(res.x, index=returns.columns)

erc_w   = erc_weights(rets)
erc_ret = (rets * erc_w).sum(axis=1)
erc_w_df = pd.DataFrame(
    np.tile(erc_w.values, (len(rets), 1)),
    index=rets.index, columns=ASSETS
)
print("ERC Risk Parity done")
print((erc_w * 100).round(2).to_string())

In [ ]:
# ── Cell 7: MPT Static ────────────────────────────────────────────────────────
# Optimizes over the FULL history — uses future data.
# In-sample only. Included to show how much look-ahead bias inflates Sharpe.

mu_arr  = (rets.mean() * 12).values
cov_arr = (rets.cov()  * 12).values

res = sco.minimize(
    lambda w: -(w @ mu_arr - RISK_FREE_ANNUAL) / np.sqrt(w @ cov_arr @ w),
    np.ones(N) / N, method="SLSQP",
    bounds=[(0, 1)] * N,
    constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1},
    options={"ftol": 1e-12, "maxiter": 1000}
)
w_sharpe   = pd.Series(res.x, index=ASSETS)
sharpe_ret = (rets * w_sharpe).sum(axis=1)
sharpe_w_df = pd.DataFrame(
    np.tile(w_sharpe.values, (len(rets), 1)),
    index=rets.index, columns=ASSETS
)
print("MPT Static done")
print((w_sharpe * 100).round(2).to_string())

In [ ]:
# ── Cell 8: MPT Rolling ───────────────────────────────────────────────────────
# Every month: optimize on the past 36 months, invest next month.
# No future data used — this is how a real robo-advisor works.
# Takes ~30 seconds.

LOOKBACK = 36

rolling_weights = []
rolling_dates   = []
rolling_returns = []

for i in range(LOOKBACK, len(rets)):
    window = rets.iloc[i - LOOKBACK : i]
    mu_w   = window.mean().values * 12
    cov_w  = window.cov().values  * 12

    # FIX: capture mu_w and cov_w with default args to avoid closure bug
    def neg_sharpe_w(w, mu=mu_w, cov=cov_w):
        r = w @ mu
        v = np.sqrt(w @ cov @ w)
        return -(r - RISK_FREE_ANNUAL) / v if v > 1e-8 else 0

    try:
        r = sco.minimize(
            neg_sharpe_w, np.ones(N) / N, method="SLSQP",
            bounds=[(0, 1)] * N,
            constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1},
            options={"ftol": 1e-10, "maxiter": 500}
        )
        w_opt = r.x
    except Exception:
        w_opt = np.ones(N) / N

    rolling_weights.append(w_opt)
    rolling_dates.append(rets.index[i])
    rolling_returns.append(w_opt @ rets.iloc[i].values)

rolling_w_df = pd.DataFrame(rolling_weights, index=rolling_dates, columns=ASSETS)
rolling_ret  = pd.Series(rolling_returns,    index=rolling_dates)

print(f"MPT Rolling done — {len(rolling_ret)} months simulated")

In [ ]:
# ── Cell 9: Performance Table ─────────────────────────────────────────────────
# Align all strategies to the rolling window period for a fair comparison.

idx = rolling_ret.index

results = pd.DataFrame([
    performance_stats(eq_ret.loc[idx],      "Equal Weight"),
    performance_stats(naive_ret.loc[idx],   "Naive Risk Parity"),
    performance_stats(erc_ret.loc[idx],     "ERC Risk Parity"),
    performance_stats(sharpe_ret.loc[idx],  "MPT Static (in-sample)"),
    performance_stats(rolling_ret,          "MPT Rolling (realistic)"),
])

print("="*62)
print("  FULL PERFORMANCE TABLE")
print(f"  Period: {idx[0].date()} -> {idx[-1].date()}")
print("="*62)
print(results.to_string())
print("="*62)

In [ ]:
# ── Cell 10: 2022 Crisis Drawdown ─────────────────────────────────────────────

crisis_2022 = pd.Series({
    "Equal Weight"           : crisis_drawdown(eq_ret,      "2022-01", "2022-12"),
    "Naive Risk Parity"      : crisis_drawdown(naive_ret,   "2022-01", "2022-12"),
    "ERC Risk Parity"        : crisis_drawdown(erc_ret,     "2022-01", "2022-12"),
    "MPT Static"             : crisis_drawdown(sharpe_ret,  "2022-01", "2022-12"),
    "MPT Rolling"            : crisis_drawdown(rolling_ret, "2022-01", "2022-12"),
}, name="2022 Max Drawdown %")

print("── 2022 Rate-Hike Crisis — Worst Drawdown ──")
print(crisis_2022.to_string())
print()
print(f"Best  protected : {crisis_2022.idxmax()}  ({crisis_2022.max()}%)")
print(f"Worst hit       : {crisis_2022.idxmin()}  ({crisis_2022.min()}%)")
print(f"Gap             : {round(crisis_2022.max() - crisis_2022.min(), 1)} percentage points")

In [ ]:
# ── Cell 11: Monthly Turnover ─────────────────────────────────────────────────
# FIX: static strategies correctly return 0% (weights never change).
# We no longer compute a ratio against 0 — just report raw numbers.
# The meaningful comparison is MPT Rolling's absolute turnover vs zero.

turnover = pd.Series({
    "Equal Weight"           : avg_turnover(eq_w_df.loc[idx]),
    "Naive Risk Parity"      : avg_turnover(naive_w_df.loc[idx]),
    "ERC Risk Parity"        : avg_turnover(erc_w_df.loc[idx]),
    "MPT Static"             : avg_turnover(sharpe_w_df.loc[idx]),
    "MPT Rolling"            : avg_turnover(rolling_w_df),
}, name="Avg Monthly Turnover %")

print("── Average Monthly Portfolio Turnover ──")
print("(0% = weights never change, 100% = full portfolio replaced each month)")
print()
print(turnover.to_string())
print()
mpt_to = turnover["MPT Rolling"]
print(f"MPT Rolling reshuffles {mpt_to}% of the portfolio every month on average.")
print("Risk Parity strategies require no monthly rebalancing (0% turnover).")

In [ ]:
# ── Cell 12: Visual Summary ───────────────────────────────────────────────────

PALETTE = {
    "Equal Weight"           : "#028090",
    "Naive Risk Parity"      : "#F6C90E",
    "ERC Risk Parity"        : "#38A169",
    "MPT Static (in-sample)" : "#6B46C1",
    "MPT Rolling (realistic)": "#E53E3E",
}
CRISES = [
    ("2008-09", "2009-06", "GFC"),
    ("2020-02", "2020-04", "COVID"),
    ("2022-01", "2022-12", "Rate Hikes"),
]
plt.rcParams.update({
    "figure.facecolor" : "#F4F7FB",
    "axes.facecolor"   : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.4,
    "grid.color"       : "#CBD5E0",
    "font.family"      : "sans-serif",
})

series_map = {
    "Equal Weight"           : eq_ret.loc[idx],
    "Naive Risk Parity"      : naive_ret.loc[idx],
    "ERC Risk Parity"        : erc_ret.loc[idx],
    "MPT Static (in-sample)" : sharpe_ret.loc[idx],
    "MPT Rolling (realistic)": rolling_ret,
}

fig, axes = plt.subplots(3, 1, figsize=(13, 14), sharex=True)

# --- Plot 1: Cumulative returns ---
ax = axes[0]
for label, ret in series_map.items():
    growth = ((1 + ret).cumprod() - 1) * 100
    ax.plot(growth, label=label, color=PALETTE[label],
            linewidth=2, linestyle="--" if "Static" in label else "-")
for start, end, label in CRISES:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.07, color="red")
# FIX: label crises after plotting so y-limits are set correctly
ymax = ax.get_ylim()[1]
for start, _, label in CRISES:
    ax.text(pd.Timestamp(start), ymax * 0.88, label, fontsize=7.5, color="#E53E3E")
ax.axhline(0, color="#8896A5", lw=0.8, ls="--")
ax.set_title("Cumulative Return", fontweight="bold", fontsize=12)
ax.set_ylabel("Return (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9)

# --- Plot 2: Drawdowns ---
ax = axes[1]
for label, ret in series_map.items():
    cum = (1 + ret).cumprod()
    dd  = ((cum - cum.cummax()) / cum.cummax()) * 100
    ax.plot(dd, label=label, color=PALETTE[label],
            linewidth=2, linestyle="--" if "Static" in label else "-")
for start, end, label in CRISES:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.07, color="red")
# FIX: label crises after plotting
ymin = ax.get_ylim()[0]
for start, _, label in CRISES:
    ax.text(pd.Timestamp(start), ymin * 0.15, label, fontsize=7.5, color="#E53E3E")
ax.axhline(0, color="#8896A5", lw=0.8)
ax.set_title("Drawdown Path", fontweight="bold", fontsize=12)
ax.set_ylabel("Drawdown (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9)

# --- Plot 3: Rolling MPT weight evolution ---
ax = axes[2]
colors = ["#028090", "#F6C90E", "#E53E3E", "#38A169", "#6B46C1", "#C05621"]
ax.stackplot(rolling_w_df.index,
             [rolling_w_df[c] * 100 for c in ASSETS],
             labels=ASSETS, colors=colors, alpha=0.85)
for start, end, _ in CRISES:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.10, color="white", zorder=2)
for start, _, label in CRISES:
    ax.text(pd.Timestamp(start), 92, label, fontsize=7.5, color="#333")
ax.set_title("MPT Rolling — How Weights Change Each Month",
             fontweight="bold", fontsize=12)
ax.set_ylabel("Weight (%)")
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9, ncol=N, loc="upper left")

fig.suptitle("Strategy Comparison", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 13: Abstract Numbers — Copy These ───────────────────────────────────
# FIX: removed the broken turnover ratio (divide-by-zero).
# Reports raw MPT turnover instead, which is the meaningful number.

erc      = results.loc["ERC Risk Parity"]
mpt_roll = results.loc["MPT Rolling (realistic)"]
mpt_stat = results.loc["MPT Static (in-sample)"]
eq       = results.loc["Equal Weight"]
naive    = results.loc["Naive Risk Parity"]

dd_2022_best  = crisis_2022.idxmax()
dd_2022_worst = crisis_2022.idxmin()
dd_2022_gap   = round(abs(crisis_2022.min() - crisis_2022.max()), 1)
mpt_static_gap = round(abs(mpt_stat["Sharpe"] - mpt_roll["Sharpe"]), 3)
mpt_turnover   = turnover["MPT Rolling"]

print("="*65)
print("  YOUR ABSTRACT NUMBERS")
print("="*65)
print(f"  Period  : {idx[0].date()} to {idx[-1].date()}")
print(f"  Assets  : {', '.join(ASSETS)}")
print()
print("  SHARPE RATIOS:")
for _, row in results.iterrows():
    print(f"    {row.name:<30} {row['Sharpe']}")
print()
print("  2022 CRISIS — WORST DRAWDOWN:")
for k, v in crisis_2022.items():
    print(f"    {k:<30} {v}%")
print(f"  → Gap between best and worst: {dd_2022_gap} percentage points")
print()
print("  TURNOVER (monthly average):")
for k, v in turnover.items():
    print(f"    {k:<30} {v}%")
print()
print("  MPT LOOK-AHEAD BIAS:")
print(f"    Static  Sharpe : {mpt_stat['Sharpe']}  (full-sample, uses future data)")
print(f"    Rolling Sharpe : {mpt_roll['Sharpe']}  (realistic, no future data)")
print(f"    Gap            : {mpt_static_gap}")
print()
print("="*65)
print("  ABSTRACT PARAGRAPH — copy and paste")
print("="*65)
print(f"""
Preliminary results, evaluated over {idx[0].strftime('%Y')}–{idx[-1].strftime('%Y')}, show
that algorithmic choice produces meaningfully different investor experiences
despite an identical asset universe. ERC Risk Parity achieved a Sharpe ratio
of {erc['Sharpe']}, compared to {mpt_roll['Sharpe']} for rolling MPT and {eq['Sharpe']} for
Equal Weight. Differences were sharpest during the 2022 rate-hike crisis,
where maximum drawdowns ranged from {crisis_2022[dd_2022_best]}% ({dd_2022_best})
to {crisis_2022[dd_2022_worst]}% ({dd_2022_worst}) — a gap of {dd_2022_gap} percentage
points. Notably, static MPT reported a Sharpe of {mpt_stat['Sharpe']} when
optimized over the full sample, but this fell to {mpt_roll['Sharpe']} under
realistic rolling re-optimization, suggesting that a portion of MPT's apparent
edge reflects look-ahead bias. Unlike MPT, Risk Parity strategies require no
monthly rebalancing ({mpt_turnover}% average monthly turnover for MPT Rolling
versus 0% for all Risk Parity variants), suggesting that simpler allocation
rules may offer a more stable investor experience without sacrificing returns.
""")
print("="*65)